In [6]:
!pip install -q langchain langchain-core langchain-google-genai google-genai

In [7]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Load API Key securely from Colab Secrets
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Initialize LLM with the active current model endpoint
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.1)

# -------------------------------------------------------------------
# SAMPLE INPUT DATA (Simulating a B2B Legal/Technical Agreement)
# -------------------------------------------------------------------
sample_agreement = """
MUTUAL NON-DISCLOSURE AGREEMENT
This Agreement is entered into on October 12, 2026 by and between TechCorp Solutions LLC ("Disclosing Party")
and Apex Innovations Inc ("Receiving Party").

1. Purpose: The parties wish to explore a potential partnership regarding AI software development.
2. Confidentiality Period: The Receiving Party shall hold confidential information for a period of 5 years from disclosure.
3. Liability: Total liability under this agreement shall not exceed $100,000 USD.
4. Termination: Either party may terminate this agreement with 30 days written notice.
5. Governing Law: This agreement shall be governed by the laws of the State of California.
"""

# -------------------------------------------------------------------
# AGENT 1: ENTITY & CLAUSE EXTRACTOR AGENT
# -------------------------------------------------------------------
extractor_prompt = ChatPromptTemplate.from_template(
    "You are an AI Information Extraction Agent.\n"
    "Analyze the following legal text and extract key structured information in bullet points:\n"
    "- Parties Involved\n"
    "- Agreement Date\n"
    "- Confidentiality Period\n"
    "- Liability Cap\n"
    "- Termination Terms\n\n"
    "Text:\n{agreement_text}"
)
extractor_chain = extractor_prompt | llm | StrOutputParser()

# -------------------------------------------------------------------
# AGENT 2: RISK & COMPLIANCE ASSESSMENT AGENT
# -------------------------------------------------------------------
risk_prompt = ChatPromptTemplate.from_template(
    "You are a Legal Risk Assessment Agent.\n"
    "Analyze the following extracted agreement details and identify any potential risks, "
    "unusual liability clauses, or compliance concerns:\n\n"
    "Extracted Details:\n{extracted_details}"
)
risk_chain = risk_prompt | llm | StrOutputParser()

# -------------------------------------------------------------------
# AGENT 3: EXECUTIVE SUMMARIZER AGENT
# -------------------------------------------------------------------
summary_prompt = ChatPromptTemplate.from_template(
    "You are an Executive Summarizer Agent.\n"
    "Combine the extracted facts and risk assessment into a clean, executive Markdown report "
    "with clear headings, bullet points, and a final risk rating (LOW, MEDIUM, or HIGH).\n\n"
    "Extracted Facts:\n{extracted_details}\n\n"
    "Risk Analysis:\n{risk_analysis}"
)
summary_chain = summary_prompt | llm | StrOutputParser()

# -------------------------------------------------------------------
# EXECUTE AGENTIC WORKFLOW (Sequential Hand-off)
# -------------------------------------------------------------------
def run_agentic_workflow(text):
    print("🤖 AGENT 1: Extracting structured information...")
    extracted_data = extractor_chain.invoke({"agreement_text": text})
    print("✅ Extraction Complete.\n")

    print("🤖 AGENT 2: Analyzing legal risk and compliance concerns...")
    risk_analysis = risk_chain.invoke({"extracted_details": extracted_data})
    print("✅ Risk Analysis Complete.\n")

    print("🤖 AGENT 3: Compiling Executive Summary Report...")
    final_report = summary_chain.invoke({
        "extracted_details": extracted_data,
        "risk_analysis": risk_analysis
    })

    print("\n" + "="*50)
    print("       FINAL AGENTIC PIPELINE OUTPUT REPORT       ")
    print("="*50 + "\n")
    print(final_report)

# Run the pipeline
run_agentic_workflow(sample_agreement)

🤖 AGENT 1: Extracting structured information...
✅ Extraction Complete.

🤖 AGENT 2: Analyzing legal risk and compliance concerns...
✅ Risk Analysis Complete.

🤖 AGENT 3: Compiling Executive Summary Report...

       FINAL AGENTIC PIPELINE OUTPUT REPORT       

# Executive Summary: Non-Disclosure Agreement Review

---

## **Overall Risk Rating: HIGH**

The agreement presents a **HIGH** risk profile, primarily driven by an aggressive $100,000 liability cap and a strict 5-year sunset on confidentiality. Under the current draft, **TechCorp Solutions LLC** faces severe exposure to unrecoverable losses and permanent trade secret forfeiture, while **Apex Innovations Inc** receives disproportionate protection with minimal liability.

---

## **1. Key Agreement Terms**

* **Disclosing Party:** TechCorp Solutions LLC  
* **Receiving Party:** Apex Innovations Inc  
* **Agreement Structure:** Unilateral (one-way disclosure)  
* **Effective Date:** October 12, 2026 *(Potential clerical error / post-